# FashionMNIST_PyTorch_Adam

이 노트북은 발표용 재실험을 위해 **한 가지 조건만 명확하게 바꾸고**,  
학습 과정과 결과를 동일한 형식으로 저장하도록 구성했습니다.

## CNN 흐름과 코드 위치
- **[1] 데이터 입력**
- **[2] 특징 추출 - Conv Block 1**
- **[3] 특징 추출 - Conv Block 2**
- **[4] 추가 특징 추출 / 분류**
- **[5] Optimizer 설정**
- **[6] 학습**
- **[7] 성능 평가**
- **[8] 결과 저장**

> Keras와 PyTorch 모두 같은 번호 체계를 사용해 PPT에서 코드 위치를 비교하기 쉽게 만들었습니다.

In [ ]:
# 공통 설정
import os, time, random, json, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
EPOCHS = 10
BATCH_SIZE = 64
LEARNING_RATE = 0.001
MOMENTUM = 0.9
EXPERIMENT_NAME = "FashionMNIST_PyTorch_Adam"

OUTPUT_DIR = Path("/content/experiment_outputs") / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = OUTPUT_DIR / "best_val_loss.pt"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Experiment:", EXPERIMENT_NAME)

## [1] 데이터 입력

Keras 실험과 동일하게 공식 Train 60,000장을 **48,000 / 12,000**으로 나눕니다.  
동일한 `random_state=42`와 stratify를 사용합니다.

In [ ]:
# [1] 데이터 입력
transform = transforms.ToTensor()  # uint8 [0,255] → float [0,1]

full_train = datasets.FashionMNIST(
    root="/content/data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="/content/data", train=False, download=True, transform=transform
)

targets = np.array(full_train.targets)
train_idx, val_idx = train_test_split(
    np.arange(len(targets)),
    test_size=12000,
    random_state=SEED,
    stratify=targets
)

train_dataset = Subset(full_train, train_idx)
val_dataset = Subset(full_train, val_idx)

generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=generator, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

CLASS_NAMES = full_train.classes

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Classes:", CLASS_NAMES)

## [2]~[4] CNN 모델 정의

Keras 실험과 **같은 CNN 구조**입니다.  
PyTorch에서는 `nn.Module` 안에서 Layer를 정의하고 `forward()`에 데이터 흐름을 직접 적습니다.

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # [2] 특징 추출 - Conv Block 1
        # Conv → BatchNorm → ReLU → Pooling
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # [3] 특징 추출 - Conv Block 2
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # [4] 분류
        # Flatten → Linear(FC) → ReLU → Dropout → Output
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        # [1] 입력 → [2][3] 특징 추출 → [4] 분류
        x = self.block1(x)
        x = self.block2(x)
        x = self.classifier(x)
        return x

In [ ]:
model = FashionCNN().to(DEVICE)
parameter_count = sum(p.numel() for p in model.parameters())
print(model)
print("Total parameters:", f"{parameter_count:,}")

## [5] Optimizer 설정

- Framework: **PyTorch**
- Optimizer: **Adam**
- Learning Rate: **0.001**


PyTorch의 `CrossEntropyLoss`는 Softmax 이전의 **logits**를 입력으로 받으므로  
모델 마지막 Layer에는 Softmax를 넣지 않습니다.

In [ ]:
# [5] Optimizer 설정
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## [6] 학습

Keras의 `fit()`과 달리 PyTorch는 아래 과정을 직접 작성합니다.

`zero_grad → forward → loss → backward → optimizer.step`

In [ ]:
# [6] 학습
# PyTorch는 Epoch / Batch / forward / backward / update 과정을 직접 작성합니다.
history_rows = []
best_val_loss = float("inf")
best_epoch = -1

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # 1) 이전 Gradient 초기화
        optimizer.zero_grad()

        # 2) Forward
        logits = model(images)

        # 3) Loss 계산
        loss = criterion(logits, labels)

        # 4) Backward
        loss.backward()

        # 5) Parameter Update
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_count += labels.size(0)

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_count = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (logits.argmax(dim=1) == labels).sum().item()
            val_count += labels.size(0)

    train_loss = train_loss_sum / train_count
    train_acc = train_correct / train_count
    val_loss = val_loss_sum / val_count
    val_acc = val_correct / val_count

    history_rows.append({
        "epoch": epoch,
        "loss": train_loss,
        "accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), BEST_MODEL_PATH)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

training_seconds = time.time() - start_time
history_df = pd.DataFrame(history_rows).set_index("epoch")

## [7] 성능 평가

In [ ]:
def predict_loader(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            logits = model(images)
            pred = logits.argmax(dim=1).cpu().numpy()
            y_pred.extend(pred)
            y_true.extend(labels.numpy())
    return np.array(y_true), np.array(y_pred)

def evaluate_predictions(y_true, y_pred, prefix):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    report = pd.DataFrame(
        classification_report(
            y_true, y_pred, target_names=CLASS_NAMES,
            output_dict=True, zero_division=0
        )
    ).T
    report.to_csv(OUTPUT_DIR / f"{prefix}_classification_report.csv", encoding="utf-8-sig")

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, xticks_rotation=45, values_format="d"
    )
    ax.set_title(f"{EXPERIMENT_NAME} - {prefix} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{prefix}_confusion_matrix.png", dpi=160, bbox_inches="tight")
    plt.show()

    return {
        f"{prefix}_test_accuracy": acc,
        f"{prefix}_macro_precision": p,
        f"{prefix}_macro_recall": r,
        f"{prefix}_macro_f1": f1
    }

# 최종 Epoch 모델 평가
final_true, final_pred = predict_loader(model, test_loader)
final_metrics = evaluate_predictions(final_true, final_pred, "final")

# Best Validation Loss 모델 평가
best_model = FashionCNN().to(DEVICE)
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
best_true, best_pred = predict_loader(best_model, test_loader)
best_metrics = evaluate_predictions(best_true, best_pred, "best")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_df.index, history_df["accuracy"], label="Train Accuracy")
plt.plot(history_df.index, history_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"{EXPERIMENT_NAME} - Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "accuracy_curve.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(history_df.index, history_df["loss"], label="Train Loss")
plt.plot(history_df.index, history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{EXPERIMENT_NAME} - Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "loss_curve.png", dpi=160, bbox_inches="tight")
plt.show()

## [8] 결과 저장

In [ ]:
history_df.to_csv(OUTPUT_DIR / "history.csv", encoding="utf-8-sig")

best_val_accuracy = float(history_df["val_accuracy"].max())

summary = {
    "experiment": EXPERIMENT_NAME,
    "dataset": "Fashion-MNIST",
    "framework": "PyTorch",
    "optimizer": "Adam",
    "model_version": "CommonCNN",
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "momentum": MOMENTUM if "Adam" == "SGD" else np.nan,
    "parameter_count": parameter_count,
    "training_seconds": training_seconds,
    "best_epoch_by_val_loss": best_epoch,
    "best_val_loss": float(best_val_loss),
    "best_val_accuracy": best_val_accuracy,
    "final_train_accuracy": float(history_df.iloc[-1]["accuracy"]),
    "final_val_accuracy": float(history_df.iloc[-1]["val_accuracy"]),
    "final_train_val_gap": float(history_df.iloc[-1]["accuracy"] - history_df.iloc[-1]["val_accuracy"]),
    "model_definition_code_lines": 27,
    "training_code_lines": 53,
    **final_metrics,
    **best_metrics
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False, encoding="utf-8-sig")
display(summary_df.T)

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Saved:", zip_path)

## [9] 결과 ZIP 자동 다운로드

Google Colab에서 실행 중이라면, 저장된 실험 결과 ZIP을 브라우저로 자동 다운로드합니다.

In [ ]:
# ------------------------------------------------------------
# [9] Colab에서 결과 ZIP 자동 다운로드
# 학습/평가/저장이 끝난 뒤 생성된 ZIP 파일을 브라우저로 다운로드합니다.
# ------------------------------------------------------------
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Google Colab 환경이 아니므로 자동 다운로드를 실행하지 않았습니다.")
    print("결과 ZIP 경로:", zip_path)